# **PRÁCTICA 1: _Construcción de un modelo URDF_**

## **_Objetivo_**
---

Construir un modelo cinemático del manipulador SO-100 a partir de las piezas STL proporcionadas, organizarlas y orientarlas en Blender, identificar las transformaciones entre sistemas de referencia y describir la estructura del robot mediante un archivo URDF. Finalmente, visualizar y verificar el modelo en VS Code mediante la extensión URDF Visualizer.

## **_Introducción_**
---

**Modelo geométrico y cinemático de un robot**

Para describir un robot manipulador en ROS, se utiliza con frecuencia el formato URDF (Unified Robot Description Format). Un archivo URDF es un documento XML que representa el robot como un conjunto de eslabones (links) conectados mediante articulaciones (joints). Cada eslabón puede incluir geometría visual, geometría de colisión y propiedades inerciales; cada junta define cómo se conecta un eslabón hijo con uno padre, el tipo de movimiento permitido y la pose relativa entre ambos sistemas de referencia.

Los tipos de junta más comunes son fixed (sin movimiento relativo), revolute (rotación limitada alrededor de un eje), continuous (rotación sin límites angulares) y prismatic (traslación). En un manipulador como el SO-100, las articulaciones de los motores se representan normalmente como articulaciones rotacionales. La etiqueta origin expresa la transformación entre los marcos de la junta padre e hijo mediante xyz (traslación) y rpy (orientación en ángulos roll, pitch y yaw, en radianes). El eje de movimiento se especifica mediante axis; por convención, se expresa en el marco de la junta.

Una descripción URDF no se obtiene únicamente colocando las mallas STL una junto a otra: es necesario determinar correctamente la relación entre los sistemas de referencia. Un error en la orientación de una junta puede hacer que el modelo se vea razonablemente bien, pero que el robot rote alrededor de un eje incorrecto. Por ello, en esta práctica se utilizará Blender para inspeccionar y preparar las piezas y, posteriormente, se describirán las transformaciones en URDF.

**Manejo básico de Blender para preparar piezas robóticas**

Blender es una herramienta de modelado 3D que permite importar mallas STL, inspeccionar su geometría, transformar objetos y establecer jerarquías padre/hijo. Cada objeto tiene un sistema de referencia local (origen del objeto y ejes locales X, Y y Z), distinto del sistema de coordenadas global de la escena. Para esta práctica, los sistemas locales deben facilitar la interpretación cinemática: el eje Z local de cada pieza asociada a un motor debe alinearse con el eje de giro de ese motor.

Las transformaciones de un objeto incluyen ubicación, rotación y escala. En Blender, Apply Rotation aplica la rotación actual a los datos de la malla y deja los valores de rotación del objeto en cero, conservando su apariencia en la escena. Esto resulta útil para preparar una geometría cuya orientación debe quedar incorporada en la malla, aunque debe hacerse deliberadamente: aplicar una rotación cambia cómo se interpretan las coordenadas locales de la geometría y puede afectar las transformaciones que luego se documenten en URDF.

En esta práctica, Blender se utilizará para construir un archivo de inspección y organización de las piezas, no para sustituir la definición cinemática del URDF. El modelo final deberá tener una jerarquía coherente y los marcos de referencia necesarios para calcular origin xyz, origin rpy y axis.



## **_Cuestionario Previo_**
---

1. ¿Cuál es la diferencia entre un link y un joint en URDF?
>En URDF, la diferencia es que el link es la parte sólida e inmóvil por sí misma (como un brazo o una rueda) que contiene la masa y el diseño visual, mientras que el joint es el mecanismo que crea el movimiento e indica cómo se mueve un link respecto al otro (si gira, se desliza o se queda fijo). En resumen: el link aporta el cuerpo físico y el joint aporta la acción y la conexión entre ellos.

2. ¿Qué información se define en las etiquetas origin, axis y parent/child de una junta URDF?
>Fijando el sistema de referencia del enlace hijo. El elemento <origin>

>Este elemento indica la ubicación y orientación del sistema de referencia L del enlace hijo. Como ya hemos comentado este sistema de referencia forma parte de la definición de la cadena cinemática y lo ubicaremos en la articulación con el enlace padre, de modo que uno de sus ejes coincida en función del tipo de articulación con el eje de giro (revolute, continuous) o con la dirección de desplazamiento (prismatic). 

>La pose del sistema de referencia L se establece en relación al sistema de referencia L del enlace padre.

>Estableciendo los enlaces de la articulación. Los elementos <parent> y <child>Estos elementos permiten proporcionar la única información que realmente es obligatoria a la hora de definir una articulación, que es la definición de los enlaces padre e hijo unidos por la misma. Indicando el eje asociado al movimiento de la articulación. El elemento <axis> Para articulaciones de tipo revolute, continuous o prismatic, el elemento <axis> permite indicar cual de los ejes del sistema de referencia L coincide con el eje de giro o de desplazamiento de la articulación.

3. ¿Cuál es la diferencia entre el sistema de coordenadas global y el sistema local de un objeto en Blender?
>La diferencia principal radica en el punto de referencia que utiliza cada sistema para medir el movimiento, la rotación y la escala de un objeto:

>• **Sistema de coordenadas global (World / Global):**

>-Tiene su origen fijo en el centro del universo 3D de la escena (el punto 0, 0, 0).

>-Los ejes X (rojo), Y (verde) y Z (azul) nunca se mueven ni cambian de orientación, sin importar hacia dónde esté volteado el objeto.

>• **Sistema de coordenadas local (Local):** 

> -Tiene su origen en el propio punto de origen del objeto (su pivote).

> -Los ejes X, Y y Z giran y se mueven junto con el objeto. Si rotas el objeto 45 grados, sus ejes locales también rotarán 45 grados

4. ¿Qué hace la operación Apply Rotation en blender?
>Estas operaciones le permiten aplicar varias transformaciones a los objetos seleccionados. Las coordenadas de transformación del objeto se transfieren a los datos del objeto. Si los objetos tienen descendientes jerárquicos, también aplica esas transformaciones a sus subordinados.

5. ¿En qué unidades se especifican las traslaciones y los ángulos en URDF?
>Las unidades de medida siguen estrictamente el Estándar del Sistema Internacional (SI):

> • **Traslaciones (\(x, y, z\)):** Se especifican en metros (m).

> • **Ángulos (rpy / roll, pitch, yaw):** Se especifican en radianes (rad).

## **_Desarrollo_**
---

**_1.  Preparación del espacio de trabajo_**

        a. Se guardo la carpeta de la practica 2 con la siguiente estructura:
```text
practica_2/
├── blender/
│   └── so100_modelo.blend
├── meshes/
│   ├── base.stl
│   └── ... (resto de las piezas proporcionadas)
├── urdf/
│   └── so100.urdf
└── imagenes/
    ├── blender_jerarquia.png
    ├── blender_ejes.png
    ├── ...
    └── urdf_visualizer.png
```
![Estructura de practica](imagenes/estructurapractica.png)

       b. Se guardo en blender un proyecto con el nombre: blender/so100_modelo.blend.
![Archivo Blender](imagenes/archivoblender.png)

        c. Se eliminaron los objetos iniciales que no se usaran.

        d. Se importaron los archivos STL mediante File → Import → STL ó arrastrándolos hacia el editor de blender desde el explorador de archivos.
![Archivos STL](imagenes/piezasimportadas.png)


**_2. Mostrar e inspeccionar los sistemas de referencia locales en Blender_**

                a. Se visualizaron los sistemas de referencia locales de cada pieza

                b. Origen local y orientación de ejes

| Pieza / eslabón | Corrección necesaria del eje "z" |
| :--- | :--- |
| **Base** | Sin Ajuste |
| **Shoulder Rotation** | 90° en X |
| **Upper Arm** | Sin Ajuste |
| **Lower Arm** | 90° en X  |
| **Wrist Pitch Roll**| Sin Ajuste |
| **Fixed Gripper**| 270° en Y|
| **Moving Jaw**| 90° en Y |


**_3. Orientar las piezas_**
        
        a. Se roto la pieza hasta que el eje Z local asociado al motor coincida con el eje físico de giro. Puede utilizar R seguido del eje global o local y un ángulo, o introducir valores numéricos en el panel N. Confirme visualmente la alineación desde más de una vista.

        b. Para que la orientación quede incorporada en la geometría, use Object → Apply → Rotation (Ctrl+A → Rotation) solo después de verificar el resultado. La rotación del objeto quedará en cero, pero la malla conservará la orientación visual. No aplique transformaciones de manera indiscriminada: registre qué piezas se modificaron y qué efecto tuvo en los ejes locales.

        c. Se Repitio el proceso para las piezas restantes. El eje Z del marco asociado a cada motor quedo alineado con el eje de giro correspondiente.

![Orientación](imagenes/orientacion.png)

**_4. Crear relaciones padre/hijo y obtener las transformaciones entre sistemas de referencia_**

        a. Se determinó la secuencia mecánica desde la base hasta el efector final. Para cada conexión, identifique cuál pieza será el padre y cuál el hijo.
        
        b. Para establecer una relación padre/hijo, se seleccionó primero el objeto hijo, después el padre.

        c. Identificación el centro del eje de giro del motor en la pieza padre.

        d. Identificación de la orientación que debe tener el marco de junta para que su eje Z coincida con el eje del motor.

        e. Se determinó la posición relativa entre el origen del marco padre y el origen de la junta.

        f. Para expresar la orientación relativa mediante ángulos roll, pitch, yaw (en ese orden y en radianes para el atributo URDF rpy). 

        g. Si el origen del STL está desplazado del eje del motor, calcule la transformación necesaria para ubicar la geometría visual respecto al marco del eslabón. Esta transformación puede ser distinta de la transformación de la junta.


| Junta | Padre | Hijo | Tipo | Origin xyz [m] | Origin rpy [rad] | axis |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | 
| Joint_1 | Base | Shoulder | Revolute | [ 0 , -0.045 , 0.016] | 0 0 0 | 0 0 1 |
| Joint_2 | Shoulder | Upper Arm | Revolute |  [ 0 , -0.0303 , 0.1203 ] | 0 0 0 | 1 0 0 |
| Joint_3 | Upper Arm | Lower Arm | Revolute |  [ 0 , 0.1126 , 0.0277 ] | 0 0 0 | 1 0 0 |
| Joint_4 | Lower Arm | Wrist | Revolute |  [ 0 , -0.135 , 0.0053 ] | 0 0 0 | 1 0 0 |
| Joint_5| Wrist  | Fixed Gripper | Revolute |  [ 0 , -0.062 , 0 ] | 0 0 0 | 0 1 0 |
| Joint_6| Fixed Gripper | Moving Jaw | Revolute |  [ 0 , -0.0245 , -0.02 ] | 0 0 0 | 1 0 0 |

![Emparentado](imagenes/Emparentado.png)

**_5. Construir el archivo URDF en Visual Studio Code_**

        a. Código

```
<robot name="so100">

<link name ="base">
    <visual>
      <origin xyz="0 0 0" rpy="0 0 0"/>
      <geometry>
        <mesh filename="../meshes/Base.STL"/>
      </geometry>
        <material name="rosaca">
            <color rgba="1.0 0.92 0.94 1.0"/>
        </material>
    </visual>
  </link>

  <link name ="shoulder">
    <visual>
      <origin xyz="0 0 0" rpy="1.571 0 0"/>
      <geometry>
        <mesh filename="../meshes/Shoulder_Rotation_Pitch.STL"/>
      </geometry>
        <material name="rosaclaro">
            <color rgba="1.0 0.71 0.75 1.0"/>
        </material>
    </visual>
  </link>

  <link name ="upper_arm">
    <visual>
      <origin xyz="0 0 0" rpy="0 0 0"/>
      <geometry>
        <mesh filename="../meshes/Upper_Arm.STL"/>
      </geometry>
        <material name="rosafuerte">
            <color rgba="1.0 0.41 0.71 1.0"/>
        </material>
    </visual>
  </link>

  <link name ="lower_arm">
    <visual>
      <origin xyz="0 0 0" rpy="1.571 0 0"/>
      <geometry>
        <mesh filename="../meshes/Lower_Arm.STL"/>
      </geometry>
        <material name="rosaotro">
            <color rgba="1.0 0.92 0.94 1.0"/>
        </material>
    </visual>
  </link>

<link name ="wrist">
    <visual>
      <origin xyz="0 0 0" rpy="0 0 0"/>
      <geometry>
        <mesh filename="../meshes/Wrist_Pitch_Roll.STL"/>
      </geometry>
        <material name="rosaclaro">
            <color rgba="1.0 0.71 0.75 1.0"/>
        </material>
    </visual>
  </link>

<link name ="fixed_gripper">
    <visual>
      <origin xyz="0 0 0" rpy="0 4.7124 0"/>
      <geometry>
        <mesh filename="../meshes/Fixed_Gripper.STL"/>
      </geometry>
        <material name="rosa">
            <color rgba="1.0 0.41 0.71 1.0"/>
        </material>
    </visual>
  </link>

<link name ="moving_jaw">
    <visual>
      <origin xyz="0 0 0" rpy="0 1.571 0"/>
      <geometry>
        <mesh filename="../meshes/Moving_Jaw.STL"/>
      </geometry>
        <material name="rosa">
            <color rgba="1.0 0.41 0.71 1.0"/>
        </material>
    </visual>
  </link>



<joint name="joint_1" type="revolute">
    <parent link="base"/>
    <child link="shoulder"/>
    <origin xyz="0 -0.045 0.016" rpy="0 0 0"/>
    <axis xyz="0 0 1"/>
    <limit lower="-1.5708" upper="1.5708" effort="10" velocity="1.0"/>
</joint>

<joint name="joint_2" type="revolute">
    <parent link="shoulder"/>
    <child link="upper_arm"/>
    <origin xyz="0 -0.0303 0.1203" rpy="0 0 0"/>
    <axis xyz="1 0 0"/>
    <limit lower="-0.349" upper="1.2217" effort="10" velocity="1.0"/>
</joint>

<joint name="joint_3" type="revolute">
    <parent link="upper_arm"/>
    <child link="lower_arm"/>
    <origin xyz="0 0.1126 0.0277" rpy="0 0 0"/>
    <axis xyz="1 0 0"/>
    <limit lower="-3.14159" upper="0" effort="10" velocity="1.0"/>
</joint>

<joint name="joint_4" type="revolute">
    <parent link="lower_arm"/>
    <child link="wrist"/>
    <origin xyz="0 -0.135 0.0053" rpy="0 0 0"/>
    <axis xyz="1 0 0"/>
    <limit lower="-1.9198" upper="1.13446" effort="10" velocity="1.0"/>
</joint>

<joint name="joint_5" type="revolute">
    <parent link="wrist"/>
    <child link="fixed_gripper"/>
    <origin xyz="0 -0.062 0.0" rpy="0 0 0"/>
    <axis xyz="0 1 0"/>
    <limit lower="-3.14159" upper="3.14159" effort="10" velocity="1.0"/>
</joint>

<joint name="joint_6" type="revolute">
    <parent link="fixed_gripper"/>
    <child link="moving_jaw"/>
    <origin xyz="0 -0.0245 -0.02" rpy="0 0 0"/>
    <axis xyz="1 0 0"/>
    <limit lower="-0.1745" upper="1.5708" effort="10" velocity="1.0"/>
</joint>


</robot>
```

**_6. Visualizar y validar el URDF con URDF Visualizer_**

        a. Visualizador

  ![Visualizador](imagenes/URDF.png)

![Visualizador Movimiento](imagenes/URDFMOV.png)



## **_Análisis de Resultados_**
---

1. ¿Qué diferencias hay entre los sistemas de referencia globales y locales de Blender? ¿Cómo los utilizó para identificar los ejes de los motores?
> El sistema global es único y fijo para toda la escena, y sus ejes X, Y y Z no cambian sin importar el objeto seleccionado. El sistema local pertenece a cada objeto: su origen está en el origen del objeto y sus ejes siguen su orientación, por lo que cambian al rotarlo. Para identificar los ejes de los motores se activó el gizmo Move en modo Local y se seleccionó cada pieza, comparando su eje Z local con el eje físico de giro del motor. Cuando no coincidían, se rotó la pieza hasta alinearlos .

>Para ubicar el centro del eje de giro desde el modo de edición y se seleccionaron los vértices del perímetro del barreno o alojamiento del motor. Con la opción Cursor to Selected (Shift+S) se colocó el cursor 3D en el centro geométrico de esa selección, que coincide con el centro del eje. Ese punto sirvió como referencia para alinear las piezas y para calcular la posición de cada junta respecto a su pieza padre.

2. ¿Qué efecto tuvo aplicar una rotación a la malla y cómo comprobó que no se alteró la interpretación cinemática de las piezas?
> Al aplicar la rotación (Object → Apply → Rotation), la orientación del objeto pasó a formar parte de los vértices de la malla y los valores de rotación del objeto quedaron en cero. La pieza se ve igual en la escena, pero sus ejes locales quedan alineados con la orientación elegida. Para comprobar que la interpretación cinemática no se alteró, se verificó que el eje Z local siguiera coincidiendo con el eje del motor y que la pieza conservara su posición y apariencia. Después se validó en el URDF Visualizer que, al mover cada junta, las piezas hijas giraran alrededor del eje esperado.

>En las piezas cuyo origen no coincidía con el eje del motor, se movió el origen del objeto al cursor 3D (Set Origin → Origin to 3D Cursor). Esto centra el sistema local en el eje de giro, pero cambia la relación entre el origen de la malla y el marco del eslabón.

3. Explique la diferencia entre la transformación definida en joint/origin y la definida en visual/origin.
> El origin de &lt;joint&gt; define la pose del marco de la junta respecto al marco del eslabón padre. Es parte de la cinemática del robot: fija dónde está el eje de giro y cómo está orientado, por lo que afecta el movimiento de toda la cadena posterior.

>El origin de &lt;visual&gt; solo coloca y orienta la malla respecto al marco de su propio eslabón. No modifica la cinemática, únicamente corrige que el origen del STL no coincida con el marco del eslabón. Por eso si una pieza se ve desplazada pero gira correctamente, el ajuste debe ser en visual/origin. Si por el otro lado, gira alrededor de un eje incorrecto, el error está en joint/origin o en axis.

4. ¿Qué ventajas tiene describir el robot mediante URDF frente a disponer únicamente de un conjunto de modelos STL?
> Los STL contienen únicamente la geometría de cada pieza, sin información de cómo se relacionan entre sí. El URDF añade la estructura del robot: eslabones, articulaciones, tipo de movimiento, ejes, límites y transformaciones entre marcos. Esto permite visualizar y simular el movimiento, calcular la cinemática directa e inversa y utilizar el modelo en herramientas como o simuladores. Además, es un formato estándar, legible y modificable, y puede ampliarse con propiedades de colisión e inercia para simulaciones dinámicas.

## **_Conclusiones_**
---
En esta práctica se construyó el modelo cinemático del manipulador SO-100 a partir de sus piezas STL, cumpliendo el objetivo de describirlo mediante un archivo URDF. El robot quedó formado por 7 eslabones conectados por 6 juntas rotacionales, que corresponden a sus 6 grados de libertad.

En Blender se inspeccionaron los sistemas de referencia locales de cada pieza, se orientaron para que el eje Z coincidiera con el eje de giro de cada motor y se organizó la jerarquía padre/hijo. Para localizar los centros de los ejes de giro se seleccionaron vértices del perímetro de los alojamientos de los motores y se usó el cursor 3D como referencia, lo que permitió determinar con mayor precisión las posiciones entre marcos. Estas transformaciones se trasladaron a los atributos origin y axis del URDF, expresando las traslaciones en metros y las orientaciones en radianes, y verificando la escala de las mallas para no compensar errores con traslaciones.

La práctica mostró que alinear visualmente las mallas no equivale a definir correctamente los marcos de junta, y que es fundamental distinguir la transformación de la junta (joint/origin) de la de la geometría visual (visual/origin). La validación en el URDF Visualizer permitió comprobar que las piezas estuvieran completas y conectadas en el orden correcto, y que cada articulación girara alrededor de su eje esperado, corrigiendo el parámetro correspondiente cuando algo no coincidía.

## **_Bibliografía_**
---